In [3]:
import pandas as pd
import pyreadstat

In [13]:
cols = (["CNT","STRATUM", "CNTSCHID", "CNTSTUID",
            "ST326Q01JA", "ESCS", "W_FSTUWT", "LANGN", "LANGTEST_COG"]
           + [f"PV{i}READ" for i in range(1, 11)] + [f"W_FSTURWT{i}" for i in range(1, 81)])

In [14]:
archivo_pisa25 = 'data/CY09_MS_STU_PUF.sav'
print("Cargando los archivo de microdatos de PISA (puede tardar unos minutos)...")
codigos_valencia25 = ['ESP33', 'ESP34']

archivo_pisa22="data/CY08MSP_STU_QQQ.SAV"
print("Cargando los archivo de microdatos de PISA 2022 (puede tardar unos minutos)...")
codigos_valencia22 = ['ESP34', 'ESP35']

archivo_pisa18="data/CY07_MSU_STU_QQQ.sav"
print("Cargando los archivo de microdatos de PISA 2018 (puede tardar unos minutos)...")
codigos_valencia18 = ['ESP9038', 'ESP1738', 'ESP9039', 'ESP1739']

def cargar_valencia(ruta, estratos, cols):
    # 1) Leemos solo las columnas que necesitamos
    df, meta = pyreadstat.read_sav(ruta, usecols=cols)

    # 2) Nos quedamos con los alumnos de España y de los estratos de Valencia
    d = df[(df["CNT"] == "ESP") & (df["STRATUM"].isin(estratos))]

    # 3) Comprobaciones: tamaño y nº de pesos replicados
    n_rep = len([c for c in d.columns if "RWT" in c])
    print("Filas:", d.shape[0], "| centros:", d["CNTSCHID"].nunique(), "| réplicas:", n_rep)
    return d, meta

df_valencia25, meta25 = cargar_valencia(archivo_pisa25, codigos_valencia25, cols)
df_valencia22, meta22 = cargar_valencia(archivo_pisa22, codigos_valencia22, cols)
df_valencia18, meta18 = cargar_valencia(archivo_pisa18, codigos_valencia18, cols)

Cargando los archivo de microdatos de PISA (puede tardar unos minutos)...
Cargando los archivo de microdatos de PISA 2022 (puede tardar unos minutos)...
Cargando los archivo de microdatos de PISA 2018 (puede tardar unos minutos)...
Filas: 1712 | centros: 54 | réplicas: 80
Filas: 1625 | centros: 53 | réplicas: 80
Filas: 1753 | centros: 53 | réplicas: 80


In [37]:
def media_pond(notas, pesos):
    return (notas * pesos).sum() / pesos.sum()

# Función porcentaje con pesos. Le paso df y el valor del idioma test que busco (depende del ciclo)
# 1ºfiltro a hispanoablantes

def pct_idioma(df, casa, prueba):
    d = df[df["LANGN"] == casa].dropna(subset=["W_FSTUWT"])   # .dropna: Elimina a aquellos estudiantes que tengan un valor nulo (NaN) en el peso muestral (W_FSTUWT)
    ceros_unos=(d["LANGTEST_COG"] == prueba).astype(float)    # 1 si prueba en valenciano, 0 si en otro
    media_ponderada=media_pond(ceros_unos, d["W_FSTUWT"])    # sumatorio(cada 1*supeso)/sumatorio(pesos)
    return 100 * media_ponderada


p_esp_val25 = pct_idioma(df_valencia25, 156, "val")    # 156: español, "val" = Valencian en 2025
p_esp_val22 = pct_idioma(df_valencia22, 156, 608)      # 156: español, 608 = Valencian en 2022
p_esp_val18 = pct_idioma(df_valencia18, 156, 608) 
print("--------PORCETAJES PONDERADOS CASTELLANOPARLANTES PRUEBA VALENCIANO--------")
print(f"En 2025:  {round(p_esp_val25, 1)}")
print(f"En 2022:  {round(p_esp_val22, 1)}")
print(f"En 2018:  {round(p_esp_val18, 1)}")

p_esp_esp25 = pct_idioma(df_valencia25, 156, "esp")    # 156: español, "val" = Valencian en 2025
p_esp_esp22 = pct_idioma(df_valencia22, 156, 156)      # 156: español, 608 = Valencian en 2022
p_esp_esp18 = pct_idioma(df_valencia18, 156, 156) 
print("--------PORCETAJES PONDERADOS CASTELLANOPARLANTES PRUEBA CASTELLANO--------")
print(f"En 2025:  {round(p_esp_esp25, 1)}")
print(f"En 2022:  {round(p_esp_esp22, 1)}")
print(f"En 2018:  {round(p_esp_esp18, 1)}")

p_val_val25 = pct_idioma(df_valencia25, 160, "val")    # 160:cat, "val" = Valencian en 2025
p_val_val22 = pct_idioma(df_valencia22, 160, 608)      # val:608, 608 = Valencian en 2022
p_val_val18 = pct_idioma(df_valencia18, 608, 608) 
print("--------PORCETAJES PONDERADOS VALENCIANOPARLANTES PRUEBA VALENCIANO--------")
print(f"En 2025:  {round(p_val_val25, 1)}")
print(f"En 2022:  {round(p_val_val22, 1)}")
print(f"En 2018:  {round(p_val_val18, 1)}")


--------PORCETAJES PONDERADOS CASTELLANOPARLANTES PRUEBA VALENCIANO--------
En 2025:  16.5
En 2022:  0.8
En 2018:  16.6
--------PORCETAJES PONDERADOS CASTELLANOPARLANTES PRUEBA CASTELLANO--------
En 2025:  83.5
En 2022:  99.1
En 2018:  83.0
--------PORCETAJES PONDERADOS VALENCIANOPARLANTES PRUEBA VALENCIANO--------
En 2025:  67.8
En 2022:  8.5
En 2018:  45.2


In [18]:
# 1) Solo los que hablan español en casa (código 156)
esp_casa = df_valencia22[df_valencia22["LANGN"] == 156]

# 2) Qué proporción hizo la prueba en cada idioma
print(esp_casa["LANGTEST_COG"].value_counts())
print((esp_casa["LANGTEST_COG"].value_counts(normalize=True) * 100).round(1))

# 3) Tabla completa, por si quieres ver todos los grupos a la vez
print(pd.crosstab(df_valencia22["LANGN"], df_valencia22["LANGTEST_COG"], margins=True))

LANGTEST_COG
156.0    1152
608.0      10
160.0       1
Name: count, dtype: int64
LANGTEST_COG
156.0    99.1
608.0     0.9
160.0     0.1
Name: proportion, dtype: float64
LANGTEST_COG  156.0  160.0  608.0   All
LANGN                                  
156.0          1152      1     10  1163
160.0             8      1      1    10
451.0             1      0      0     1
474.0             1      0      0     1
608.0           220      6     57   283
668.0             1      0      0     1
852.0           140      0      2   142
999.0            24      0      0    24
All            1547      8     70  1625


In [20]:
etq_test = meta25.variable_value_labels.get("LANGTEST_COG", {})
etq_casa = meta25.variable_value_labels.get("LANGN", {})

print({k: v for k, v in etq_test.items() if k in (156.0, 160.0, 608.0)})
print({k: v for k, v in etq_casa.items() if k in (156.0, 160.0, 608.0, 852.0, 668.0, 999.0)})

{}
{156.0: 'Spanish (spa)', 160.0: 'Catalan (cat)', 999.0: 'Missing (99)'}


In [17]:
# En cuántos idiomas distintos se examinó el alumnado de cada centro
print(df_valencia25.groupby("CNTSCHID")["LANGTEST_COG"].nunique().value_counts())

LANGTEST_COG
1    46
2     8
Name: count, dtype: int64


In [29]:
df_valencia18["LANGTEST_COG"].value_counts()

LANGTEST_COG
156.0    1329
608.0     413
160.0      11
Name: count, dtype: int64

In [27]:
por_centro = df_valencia25.groupby("CNTSCHID").agg(
    alumnos=("CNTSTUID", "size"),
    pct_val=("LANGTEST_COG", lambda s: (s == "val").mean() * 100),
    estrato=("STRATUM", "first"),
)

# Clasificamos cada centro según el idioma en que examinó a su alumnado
por_centro["tipo"] = pd.cut(por_centro["pct_val"], bins=[-1, 0, 99.999, 100],
                            labels=["Solo español", "Mixto", "Solo valenciano"])

print(por_centro["tipo"].value_counts())                      # cuántos centros de cada tipo
print(pd.crosstab(por_centro["estrato"], por_centro["tipo"]))  # público/privado según tipo
print(por_centro.groupby("tipo")["alumnos"].sum())             # cuántos alumnos hay en cada tipo

tipo
Solo español       40
Mixto               8
Solo valenciano     6
Name: count, dtype: int64
tipo     Solo español  Mixto  Solo valenciano
estrato                                      
ESP33              21      8                6
ESP34              19      0                0
tipo
Solo español       1246
Mixto               274
Solo valenciano     192
Name: alumnos, dtype: int64


In [ ]:
# -----------------------------PRUEBAS-------------------------------

# Ver la etiqueta descriptiva que SPSS le da a cada código de estrato en 2018
etiquetas_estrato = meta25.variable_value_labels.get("STRATUM")

if etiquetas_estrato:
  for codigo in ["ESP33", "ESP34"]:  # O prueba los que quieras consultar
    print(f"Código {codigo}: {etiquetas_estrato.get(codigo, 'No encontrado')}")
else:
  print("No se encontraron etiquetas de valor integradas para STRATUM.")

Código ESP33: Comunidad Valenciana, Public
Código ESP34: Comunidad Valenciana, Private


In [17]:
etiquetas_estrato = meta25.variable_value_labels.get("STRATUM")

if etiquetas_estrato:
  print(
      "Buscando 'Valencia' o 'Valenciana' en los metadatos de STRATUM (2022)..."
  )
  encontrados = False
  for codigo, etiqueta in etiquetas_estrato.items():
    # Comprobamos si el texto de la etiqueta contiene valencia (sin importar mayúsculas/minúsculas)
    if any(
        palabra in str(etiqueta).lower() for palabra in ["valencia", "valenciana"]
    ):
      print(f" Código exacto: {codigo}  -->  Etiqueta: {etiqueta}")
      encontrados = True

  if not encontrados:
    print(
        "No se ha encontrado ninguna coincidencia con 'Valencia' en las"
        " etiquetas de STRATUM."
    )
else:
  print("No se encontraron etiquetas de valor integradas para STRATUM.")

Buscando 'Valencia' o 'Valenciana' en los metadatos de STRATUM (2022)...
 Código exacto: ESP33  -->  Etiqueta: Comunidad Valenciana, Public
 Código exacto: ESP34  -->  Etiqueta: Comunidad Valenciana, Private
